# Re-ranking

**Difficulty:** Beginner

Fast keyword search narrows thousands of documents to a short list fast, but rarely puts the right one first. Re-ranking scores each shortlisted candidate against the query individually, with one `Noul` per pair, and sorts by that score instead.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/rerank_typesafe (click to expand)</summary>

# Re-ranking

> Builds 30-passage BM25 shortlists for 40 CLERC legal queries, then uses one TypeSafe question per query-candidate pair to raise top-1 accuracy from 5% to 18% and top-10 accuracy from 38% to 62%.

## How do we find one document in thousands?

Checking every document against a query one at a time works, but doesn't scale: millions of documents means millions of comparisons per query. A two-step approach helps:

1. Cut the pile down to a shortlist of plausible candidates, using a method fast enough to run on the whole pile (**fast search** — e.g. BM25 keyword matching, or dense embeddings).
2. Apply a more accurate step to that shortlist to find the exact right answer (**re-ranking**).

Fast search is good at narrowing the field, but bad at telling you which shortlisted candidate is actually correct. Re-ranking compares the query against each shortlisted candidate individually, and sorts by that score.

## Re-ranking with TypeSafe

A re-ranker needs a comparable score for every query-candidate pair. The scoring request can be a plain yes/no question — a `Noul` — asking something like *"could this candidate passage be from the cited precedent?"* The `noul` returned (0 to 1) is the score the application sorts on:

```
nouls = {candidate: ask_typesafe(query, candidate) for candidate in shortlist}
reranked = sorted(shortlist, key=lambda c: nouls[c], reverse=True)  # highest noul first
```

One request per candidate, none of them seeing another — the same speculative-fan-out shape you've already used, just applied to "score every candidate" instead of "answer every question."

## The worked example: 3,565 US court opinion passages, 40 queries

BM25 builds a fast-search shortlist of 30 candidates for each of 40 legal queries from the CLERC dataset. Fast search alone puts the correct passage in the shortlist for **100%** of queries — but ranks it *first* on that shortlist only **5%** of the time. TypeSafe then re-ranks each 30-candidate shortlist: one `Noul` question per (query, candidate) pair, 1,200 calls total, asking whether the candidate could be the passage the query's removed citation points to:

```
Noul(
    instructions="...Could the candidate passage be from that cited precedent — does it "
                 "establish the specific legal proposition the query excerpt invokes at "
                 "its citation point?",
    criteria={
        "true": "The candidate passage states or establishes the specific rule, standard, "
                "holding, or fact pattern that the query excerpt attributes to its removed citation.",
        "false": "The candidate passage is merely on a similar topic or doctrine; it does not "
                 "supply the specific proposition the query excerpt relies on.",
    },
)
```

## Re-ranking moves the right answer toward the top

| | Top 1 | Top 5 | Top 10 |
| --- | --- | --- | --- |
| Fast search alone | 5% | 15% | 38% |
| + TypeSafe re-rank | 18% | 35% | 62% |

All 1,200 calls together used 1,536,002 input and 25,200 output tokens, costing $0.0645 total (historical pricing, not current billing).

> The full cookbook builds the BM25 shortlists from a real 3,565-passage legal corpus and renders bar charts of the before/after accuracy — trimmed here to the technique itself, since building a BM25 index isn't this repo's purpose. Read the full cookbook at [docs.typesafe.ai/cookbooks/rerank_typesafe](https://docs.typesafe.ai/cookbooks/rerank_typesafe) any time.

</details>

## The core idea

There's no new primitive here — re-ranking is just `Noul` (lesson 4) aimed at a specific job: "does this candidate actually answer this query?", asked once per candidate, sorted by the result. What makes it worth naming as its own technique is what it *isn't*: it doesn't need a scoring scale invented for a general-purpose model, it doesn't rank the whole shortlist in one call (which would need the model to compare candidates against each other, not just against the query), and it doesn't touch fast search's job of narrowing the field in the first place. Fast search finds *candidates*; re-ranking finds the *right one among them*.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { noul } from "npm:@typesafe-ai/sdk@^0.7.0";

const query = "What is the deadline for reporting a data breach under GDPR?";

const candidates = [
  "Article 33 of the GDPR requires controllers to notify the supervisory authority of a personal data breach without undue delay and, where feasible, within 72 hours of becoming aware of it.",
  "The GDPR grants data subjects the right to erasure, commonly known as the right to be forgotten, allowing individuals to request deletion of their personal data.",
  "Data protection officers must be appointed by public authorities and organisations engaged in large-scale monitoring or processing of sensitive data.",
  "Administrative fines under the GDPR can reach up to EUR 20 million or 4% of a company's global annual turnover, whichever is higher.",
  "The GDPR applies to any organisation processing personal data of EU residents, regardless of where the organisation itself is established.",
];

const isRelevant = noul(
  "Does the candidate passage state the specific rule or fact the query is asking about?",
  {
    true: "The candidate directly answers the query's specific question.",
    false: "The candidate is only on a related topic, but doesn't answer this specific question.",
  },
);

const scored = await Promise.all(
  candidates.map(async (candidate) => {
    const response = await client.systemOne({
      state: { query, candidate },
      questions: { isRelevant },
    });
    return { candidate, noul: response.answers.isRelevant.noul };
  }),
);

scored.sort((a, b) => b.noul - a.noul);

for (const { candidate, noul: score } of scored) {
  console.log(score.toFixed(2), "-", candidate.slice(0, 70) + "...");
}

## What just happened

Five candidates, each scored independently against the same query, sorted by `noul`. The 72-hour breach-notification passage should land on top — it's the only one that actually answers "what is the deadline." The others are all genuinely GDPR-related (a keyword search would happily shortlist every one of them), which is exactly the case re-ranking exists for: fast search got the topic right, re-ranking has to get the *specific answer* right.

**Try it yourself:** add a sixth candidate that mentions "72 hours" in an unrelated context (e.g. a different deadline entirely) and predict whether it fools the ranking — a real test of whether the `Noul` is reading for the specific proposition or just pattern-matching on shared words.

## Related

- `lessons/core-concepts/0004-noul-and-picking-the-right-primitive.ipynb` — the `Noul` primitive this technique is built on
- `lessons/patterns/0006-speculative-fan-out.ipynb` — the "one independent call per candidate" shape
- `cookbooks/how-to/line-by-line-search.ipynb` — a related way to search a corpus by meaning

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.